# Fisheye Image Undistortion Inference

This notebook loads a trained model and performs inference to undistort fisheye images.
- Load checkpoint from training
- Apply undistortion to input images
- Plot and save results

In [ ]:
# Install required dependencies if needed
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install opencv-python-headless pillow scipy pyyaml matplotlib tqdm -q

In [ ]:
# Clone repository if not already done
import os
if not os.path.exists('/kaggle/working/SelfSupervisedFisheyeRectification'):
    os.chdir('/kaggle/working')
    !git clone https://github.com/memara111/SelfSupervisedFisheyeRectification.git
os.chdir('/kaggle/working/SelfSupervisedFisheyeRectification')

In [ ]:
# Setup paths
CHECKPOINT_PATH = '/kaggle/working/outputs/nyu_depth_v2/checkpoint.pth.tar'
INPUT_IMAGE_PATH = '/kaggle/input/your-images'  # Change this to your image directory
OUTPUT_DIR = '/kaggle/working/undistorted_images'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Checkpoint path: {CHECKPOINT_PATH}')
print(f'Input directory: {INPUT_IMAGE_PATH}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Import modules
import sys
sys.path.insert(0, '/kaggle/working/SelfSupervisedFisheyeRectification/src')

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import cv2

from models import ParametersEstimationModule
from datasets import FisheyeEffector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {DEVICE}')

In [ ]:
# Initialize model
in_channels = 3
model = ParametersEstimationModule(in_channels=in_channels).to(DEVICE)
transform = model.getTransforms()

if DEVICE == 'cuda':
    model = torch.nn.DataParallel(model)

model = model.eval()
print('Model initialized')

In [ ]:
# Load checkpoint
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    if 'state_dict' in checkpoint.keys():
        model.load_state_dict(checkpoint['state_dict'], strict=False)
    else:
        model.load_state_dict(checkpoint, strict=False)
    print(f'=> Loaded checkpoint from epoch {checkpoint.get("epoch", "N/A")}')
else:
    print(f'ERROR: Checkpoint not found at {CHECKPOINT_PATH}')
    print('Please train the model first using the training notebook.')

In [ ]:
# Function to undistort an image
def undistort_image(model, image_path, output_height=256, output_width=512):
    """
    Predict distortion and undistort an image.
    
    Args:
        model: Trained model
        image_path: Path to input image
        output_height: Height of output image
        output_width: Width of output image
    
    Returns:
        original_img: Original PIL image
        undistorted_img: Undistorted PIL image
        predicted_distortion: Predicted distortion value
    """
    # Load and transform image
    original_img = Image.open(image_path).convert('RGB')
    original_size = original_img.size
    
    # Resize for model input
    input_img = original_img.resize((output_width, output_height))
    input_tensor = transform(input_img).unsqueeze(0).to(DEVICE)
    
    # Predict distortion
    with torch.no_grad():
        predicted_distortion = model(input_tensor)
        distortion_value = predicted_distortion.data.to('cpu').item()
    
    print(f'Predicted distortion: {distortion_value:.4f}')
    
    # Create effector with negative distortion to undo the fisheye effect
    # The model predicts the distortion that was applied, so we use negative to reverse it
    effector = FisheyeEffector(
        height=output_height,
        width=output_width,
        distortion=-distortion_value  # Negative to undo distortion
    )
    
    # Apply rectification
    undistorted_img = effector(input_img)
    
    return original_img, undistorted_img, distortion_value

In [ ]:
# Find images to process
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
image_files = []

if os.path.exists(INPUT_IMAGE_PATH):
    for file in os.listdir(INPUT_IMAGE_PATH):
        if any(file.lower().endswith(ext) for ext in image_extensions):
            image_files.append(os.path.join(INPUT_IMAGE_PATH, file))
    
    print(f'Found {len(image_files)} images to process')
    for img_path in image_files[:5]:
        print(f'  - {img_path}')
    if len(image_files) > 5:
        print(f'  ... and {len(image_files) - 5} more')
else:
    print(f'Input directory not found: {INPUT_IMAGE_PATH}')
    print('Please upload your images or change the INPUT_IMAGE_PATH variable.')

In [ ]:
# Process and display images
output_height = 256
output_width = 512

for idx, img_path in enumerate(image_files):
    print(f'\nProcessing image {idx+1}/{len(image_files)}: {os.path.basename(img_path)}')
    print('-' * 60)
    
    try:
        original, undistorted, distortion = undistort_image(
            model, 
            img_path, 
            output_height=output_height, 
            output_width=output_width
        )
        
        # Save undistorted image
        output_filename = f'undistorted_{os.path.basename(img_path)}'
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        undistorted.save(output_path)
        print(f'Saved: {output_path}')
        
        # Plot comparison
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        axes[0].imshow(original)
        axes[0].set_title(f'Original Image\n(Distortion: {distortion:.4f})', fontsize=12)
        axes[0].axis('off')
        
        axes[1].imshow(undistorted)
        axes[1].set_title('Undistorted Image', fontsize=12)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'comparison_{os.path.basename(img_path)}.png'), dpi=150)
        plt.show()
        
    except Exception as e:
        print(f'Error processing {img_path}: {str(e)}')
        import traceback
        traceback.print_exc()

In [ ]:
# Optional: Test with different distortion values manually
# This is useful if you want to experiment with different levels of correction

def apply_manual_undistortion(image_path, distortion_value, output_height=256, output_width=512):
    """
    Apply manual undistortion with a specified distortion value.
    
    Args:
        image_path: Path to input image
        distortion_value: Distortion value (positive for barrel, negative for pincushion)
        output_height: Output height
        output_width: Output width
    """
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize((output_width, output_height))
    
    effector = FisheyeEffector(
        height=output_height,
        width=output_width,
        distortion=distortion_value
    )
    
    result = effector(img_resized)
    return img, result

# Example: Try different distortion values on the first image
if len(image_files) > 0:
    test_img = image_files[0]
    print(f'Testing manual undistortion on: {os.path.basename(test_img)}')
    
    distortion_values = [-0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3]
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Show original
    original = Image.open(test_img)
    axes[0].imshow(original)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    
    # Show different distortion corrections
    for i, dist_val in enumerate(distortion_values):
        try:
            orig, corrected = apply_manual_undistortion(test_img, dist_val)
            axes[i+1].imshow(corrected)
            axes[i+1].set_title(f'Distortion: {dist_val:.2f}', fontsize=12)
            axes[i+1].axis('off')
        except Exception as e:
            axes[i+1].text(0.5, 0.5, f'Error:\n{str(e)}', ha='center', va='center')
            axes[i+1].axis('off')
    
    # Empty last subplot
    axes[-1].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'manual_distortion_comparison.png'), dpi=150)
    plt.show()

In [ ]:
# Summary
print('\n' + '='*60)
print('INFERENCE COMPLETE')
print('='*60)
print(f'Processed {len(image_files)} images')
print(f'Output directory: {OUTPUT_DIR}')
print(f'\nFiles saved:')
for file in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, file)
    file_size = os.path.getsize(file_path)
    print(f'  - {file} ({file_size:,} bytes)')